In [1]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="llama3.1",
    # temperature=0,
    # other params...
)

In [3]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi! I'm Bob")])

AIMessage(content="Nice to meet you, Bob! How's your day going so far? Is there something on your mind that you'd like to chat about, or are you just looking for a friendly conversation?", additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2024-09-27T14:56:12.763771117Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 5490177517, 'load_duration': 4875006467, 'prompt_eval_count': 15, 'prompt_eval_duration': 27892000, 'eval_count': 40, 'eval_duration': 542366000}, id='run-d7c71a77-d3b5-41b1-90ef-a221090ac42e-0', usage_metadata={'input_tokens': 15, 'output_tokens': 40, 'total_tokens': 55})

In [4]:
model.invoke([HumanMessage(content="What's my name?")])

AIMessage(content="I'm a large language model, I don't have any information about your personal details, including your name. Our conversation just started, and I'm here to help with any questions you may have. If you'd like to share your name with me, I'd be happy to know it!", additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2024-09-27T14:56:29.452881478Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 957304216, 'load_duration': 61749758, 'prompt_eval_count': 15, 'prompt_eval_duration': 31755000, 'eval_count': 60, 'eval_duration': 819583000}, id='run-4f4ec014-1f0d-4f7b-92ed-4bbf2d75591c-0', usage_metadata={'input_tokens': 15, 'output_tokens': 60, 'total_tokens': 75})

# 加入聊天历史

In [5]:
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content="Hi! I'm Bob"),
        AIMessage(content="Hello Bob! How can I assist you today?"),
        HumanMessage(content="What's my name?"),
    ]
)

AIMessage(content="Your name is Bob. We just established that! How's your day going so far, Bob?", additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2024-09-27T15:01:58.660691127Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 5315262522, 'load_duration': 4845007678, 'prompt_eval_count': 40, 'prompt_eval_duration': 38209000, 'eval_count': 21, 'eval_duration': 301006000}, id='run-b0817448-bc35-40d2-85ea-59e0113a83a9-0', usage_metadata={'input_tokens': 40, 'output_tokens': 21, 'total_tokens': 61})

# 消息历史记录

In [8]:
from langchain_core.chat_history import (
    BaseChatMessageHistory,
    InMemoryChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [9]:
config = {"configurable": {"session_id": "abc2"}}

In [10]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi! I'm Bob")],
    config=config,
)

response.content

'Hello Bob! Nice to meet you. Is there something I can help you with or would you like to chat for a bit?'

In [11]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

"Your name is Bob! We just established that! How's your day going so far, Bob?"

In [12]:
config = {"configurable": {"session_id": "abc3"}}

response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'I\'m a large language model, I don\'t have the ability to know or remember information about individual users unless it\'s provided within our conversation. So, I don\'t actually "know" your name.\n\nHowever, if you\'d like to tell me your name, I can happily chat with you and use it in our conversation!'

In [13]:
config = {"configurable": {"session_id": "abc2"}}

response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

"Your name is Bob. I've already told you that twice!"

# 提示模板

In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [15]:
response = chain.invoke({"messages": [HumanMessage(content="hi! I'm bob")]})

response.content

"Hi Bob! It's great to meet you! Is there something I can help you with, or would you like to chat for a bit?"

In [16]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [17]:
config = {"configurable": {"session_id": "abc5"}}

In [18]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi! I'm Jim")],
    config=config,
)

response.content

"Hello Jim! It's nice to meet you. Is there something I can help you with or would you like to chat?"

In [19]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

"Your name is Jim! We just established that, didn't we?"

In [20]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [21]:
response = chain.invoke(
    {"messages": [HumanMessage(content="hi! I'm bob")], "language": "Chinese"}
)

response.content

'你好！我也喜欢叫你Bob！很高兴见到你，怎么可以帮你的忙？'